# 01. Calibración de confianza de los modelos RetiScan

Este notebook calibra las salidas de los modelos seleccionados para
Retinopatía Diabética y Glaucoma.

Los modelos ya fueron entrenados, comparados y congelados previamente.

La calibración no modifica los pesos de las redes neuronales ni cambia
las arquitecturas seleccionadas. Su objetivo es transformar los scores
de salida en valores más adecuados para su interpretación como confianza
del modelo.

Se utiliza exclusivamente el conjunto Validation para ajustar los
calibradores.

Modelos seleccionados:

- Retinopatía Diabética: EfficientNetB3+SE con Fine-Tuning.
- Glaucoma: ResNet50V2 con Fine-Tuning.

El conjunto Test no se utiliza para ajustar la calibración.

## 1. Importación de dependencias y rutas de resultados

Se importan las librerías para optimización matemática (`minimize_scalar`), métricas de calibración probabilística (`log_loss`, `brier_score_loss`) y manipulación tabular, configurando los directorios de salida y verificando la existencia de las predicciones de Validation de ambos modelos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    log_loss,
    brier_score_loss
)

PROJECT_ROOT = Path(
    "/mnt/c/Users/SeuNg720/Documents/TP1-RetiScan"
)
CALIBRATION_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "calibration"
)
CALIBRATION_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RD_VALIDATION_PATH = (
    PROJECT_ROOT
    / "results"
    / "RD"
    / "efficientnetb3_se"
    / "efficientnetb3_se_finetune_validation_predictions.csv"
)
GLAUCOMA_VALIDATION_PATH = (
    PROJECT_ROOT
    / "results"
    / "glaucoma"
    / "model_comparison"
    / "resnet50v2_validation_scores.csv"
)

print(
    "RD Validation:",
    RD_VALIDATION_PATH.exists()
)
print(
    "Glaucoma Validation:",
    GLAUCOMA_VALIDATION_PATH.exists()
)

RD Validation: True
Glaucoma Validation: True


## 2. Carga y verificación de predicciones de Retinopatía Diabética

Se leen las predicciones de Validation del modelo seleccionado EfficientNetB3+SE, comprobando los 523 registros y la estructura de los scores Softmax por clase.

In [2]:
rd_val_df = pd.read_csv(
    RD_VALIDATION_PATH
)

print(
    "Registros RD:",
    len(rd_val_df)
)
print(
    "Columnas:"
)
print(
    rd_val_df.columns.tolist()
)
display(
    rd_val_df.head()
)

Registros RD: 523
Columnas:
['id_code', 'diagnosis', 'predicted_class', 'score_class_0', 'score_class_1', 'score_class_2', 'score_class_3', 'score_class_4']


,id_code,diagnosis,predicted_class,score_class_0,score_class_1,score_class_2,score_class_3,score_class_4
0,0024cdab0c1e,1,1,0.006641,0.920486,0.057600,0.001871,0.013401
1,0083ee8054ee,4,2,0.000063,0.122010,0.578128,0.190302,0.109497
2,00a8624548a9,2,2,0.000030,0.351160,0.538882,0.029997,0.079931
3,00b74780d31d,2,2,0.000628,0.140549,0.518544,0.255520,0.084758
4,01b3aed3ed4c,1,1,0.001004,0.801254,0.184329,0.004688,0.008725


## 3. Inspección de las salidas de Validation para Glaucoma

Se cargan los scores producidos por ResNet50V2 sobre el conjunto
Validation utilizado durante la selección experimental.

El objetivo es identificar la estructura exacta de las salidas antes de
realizar la calibración binaria.

In [3]:
glaucoma_val_df = pd.read_csv(
    GLAUCOMA_VALIDATION_PATH
)
print(
    "Registros Glaucoma:",
    len(glaucoma_val_df)
)
print(
    "Columnas:"
)
print(
    glaucoma_val_df.columns.tolist()
)
display(
    glaucoma_val_df.head()
)

Registros Glaucoma: 202
Columnas:
['image_name', 'label', 'resnet50v2_score']


,image_name,label,resnet50v2_score
0,IMG_2544.JPG,0,0.006071
1,IMG_3380.JPG,0,0.000201
2,IMG_3506.JPG,0,0.021984
3,IMG_3625.JPG,0,0.014642
4,Image188.JPG,1,0.438085


## 4. Verificación de las propiedades probabilísticas de RD

Se comprueba que los scores generados por EfficientNetB3+SE satisfagan las propiedades formales de una distribución Softmax (sumatoria unitaria) y correspondan exactamente al vector de predicciones `predicted_class`.

In [4]:
RD_SCORE_COLUMNS = [
    "score_class_0",
    "score_class_1",
    "score_class_2",
    "score_class_3",
    "score_class_4"
]
rd_scores = (
    rd_val_df[
        RD_SCORE_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)
rd_labels = (
    rd_val_df[
        "diagnosis"
    ]
    .to_numpy(
        dtype=np.int32
    )
)

score_sums = rd_scores.sum(
    axis=1
)
argmax_classes = np.argmax(
    rd_scores,
    axis=1
)

print(
    "Shape scores:",
    rd_scores.shape
)
print(
    "Suma mínima:",
    score_sums.min()
)
print(
    "Suma máxima:",
    score_sums.max()
)
print(
    "Predicted_class coincide con argmax:",
    np.array_equal(
        argmax_classes,
        rd_val_df[
            "predicted_class"
        ].to_numpy()
    )
)

Shape scores: (523, 5)
Suma mínima: 0.999999853548
Suma máxima: 1.0000001527740001
Predicted_class coincide con argmax: True


## 5. Calibración de Retinopatía Diabética mediante Temperature Scaling

Se utiliza Temperature Scaling para calibrar las salidas multiclase de
EfficientNetB3+SE.

Se aprende un único parámetro positivo `T` utilizando exclusivamente
Validation.

- `T = 1`: no modifica las salidas.
- `T > 1`: reduce predicciones excesivamente confiadas.
- `T < 1`: incrementa la concentración de las probabilidades.

La clase predicha no cambia, ya que Temperature Scaling preserva el orden
de los logits.

### 5.1. Funciones auxiliares para recuperación de logits y escalamiento

Se definen funciones numéricamente estables para transformar probabilidades a logits equivalentes (`log(scores)`), computar Softmax con desplazamiento de escala y reponderar por temperatura.

In [5]:
EPSILON = 1e-7


def probabilities_to_logits(
    probabilities
):
    probabilities = np.clip(
        probabilities,
        EPSILON,
        1.0
    )
    return np.log(
        probabilities
    )


def softmax_numpy(
    logits
):
    shifted = (
        logits
        - np.max(
            logits,
            axis=1,
            keepdims=True
        )
    )
    exp_values = np.exp(
        shifted
    )
    return (
        exp_values
        / exp_values.sum(
            axis=1,
            keepdims=True
        )
    )


def apply_temperature(
    logits,
    temperature
):
    return softmax_numpy(
        logits / temperature
    )

### 5.2. Reconstrucción y verificación de logits equivalentes

Se convierten los scores Softmax a logits y se verifica la reversibilidad exacta del cálculo.

In [6]:
rd_logits = probabilities_to_logits(
    rd_scores
)
rd_reconstructed_scores = softmax_numpy(
    rd_logits
)

print(
    "Reconstrucción correcta:",
    np.allclose(
        rd_scores,
        rd_reconstructed_scores,
        atol=1e-6
    )
)

Reconstrucción correcta: True


### 5.3. Optimización de temperatura por minimización de pérdida logarítmica (NLL)

Se define la función objetivo basada en la pérdida logarítmica multiclase (*Cross-Entropy / Negative Log-Likelihood*) evaluada en Validation y se halla la temperatura óptima `T` mediante minimización acotada.

In [7]:
def temperature_objective(
    temperature
):
    calibrated_scores = apply_temperature(
        rd_logits,
        temperature
    )
    return log_loss(
        rd_labels,
        calibrated_scores,
        labels=[0, 1, 2, 3, 4]
    )


temperature_result = minimize_scalar(
    temperature_objective,
    bounds=(
        0.05,
        10.0
    ),
    method="bounded"
)
RD_TEMPERATURE = float(
    temperature_result.x
)

print(
    "Optimización exitosa:",
    temperature_result.success
)
print(
    "Temperatura óptima:",
    RD_TEMPERATURE
)

Optimización exitosa: True
Temperatura óptima: 1.1019132499924953


### 5.4. Aplicación de la temperatura óptima y verificación de invariancia de clases

Se calibran las probabilidades con `RD_TEMPERATURE` y se corrobora que la clase predicha permanezca estrictamente idéntica para todas las observaciones.

In [8]:
rd_calibrated_scores = apply_temperature(
    rd_logits,
    RD_TEMPERATURE
)
rd_calibrated_prediction = np.argmax(
    rd_calibrated_scores,
    axis=1
)
rd_calibrated_confidence = np.max(
    rd_calibrated_scores,
    axis=1
)

print(
    "Las clases predichas permanecen iguales:",
    np.array_equal(
        argmax_classes,
        rd_calibrated_prediction
    )
)
print(
    "Confianza calibrada mínima:",
    rd_calibrated_confidence.min()
)
print(
    "Confianza calibrada máxima:",
    rd_calibrated_confidence.max()
)

Las clases predichas permanecen iguales: True
Confianza calibrada mínima: 0.29256778381211507
Confianza calibrada máxima: 0.9997385940668643


### 5.5. Evaluación del impacto de la calibración sobre la pérdida logarítmica

Se compara cuantitativamente la pérdida logarítmica (Negative Log-Likelihood) de Validation antes y después del escalamiento por temperatura.

In [9]:
rd_nll_before = log_loss(
    rd_labels,
    rd_scores,
    labels=[0, 1, 2, 3, 4]
)
rd_nll_after = log_loss(
    rd_labels,
    rd_calibrated_scores,
    labels=[0, 1, 2, 3, 4]
)

print(
    f"NLL antes:   {rd_nll_before:.6f}"
)
print(
    f"NLL después: {rd_nll_after:.6f}"
)
print(
    f"Temperatura: {RD_TEMPERATURE:.6f}"
)

NLL antes:   0.576958
NLL después: 0.574933
Temperatura: 1.101913


/home/seung720/.venvs/retiscan/lib/python3.12/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


## 6. Persistencia del calibrador de Retinopatía Diabética

Se exportan los parámetros y la metadata del calibrador Temperature Scaling a un archivo JSON en `results/calibration/rd_temperature_scaling.json` para su posterior carga en el pipeline de inferencia clínica.

In [10]:
import json

RD_CALIBRATION_PATH = (
    CALIBRATION_RESULTS_DIR
    / "rd_temperature_scaling.json"
)

rd_calibration_config = {
    "task": "diabetic_retinopathy",
    "model": "EfficientNetB3+SE Fine-Tuning",
    "method": "temperature_scaling",
    "temperature": float(RD_TEMPERATURE),
    "validation_samples": int(len(rd_labels)),
    "nll_before": float(rd_nll_before),
    "nll_after": float(rd_nll_after),
    "classes": {
        "0": "Sin indicios de Retinopatía Diabética",
        "1": "Leve",
        "2": "Moderada",
        "3": "Severa",
        "4": "Proliferativa"
    }
}

with open(
    RD_CALIBRATION_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        rd_calibration_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Calibrador RD guardado en:",
    RD_CALIBRATION_PATH
)

Calibrador RD guardado en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/calibration/rd_temperature_scaling.json


## 7. Calibración de Glaucoma mediante Platt Scaling

La salida de ResNet50V2 corresponde a un score sigmoid asociado a la
categoría Alto riesgo de Glaucoma.

Se utiliza una calibración logística tipo Platt Scaling ajustada
exclusivamente sobre Validation.

La calibración transforma el score bruto en una probabilidad calibrada
de Alto riesgo.

El umbral operativo previamente seleccionado no se modifica. En su
lugar, dicho umbral se transforma mediante el mismo calibrador, de forma
que las decisiones Bajo/Alto riesgo permanezcan exactamente iguales.

### 7.1. Carga y verificación de scores y umbral operativo de Glaucoma

Se extraen las 202 observaciones de Validation, las etiquetas binarias (175 bajo riesgo, 27 alto riesgo) y el umbral de decisión congelado previamente (`0.100155`).

In [11]:
from scipy.optimize import minimize
from scipy.special import expit

glaucoma_labels = (
    glaucoma_val_df["label"]
    .to_numpy(dtype=np.int32)
)
glaucoma_scores = (
    glaucoma_val_df["resnet50v2_score"]
    .to_numpy(dtype=np.float64)
)
GLAUCOMA_RAW_THRESHOLD = 0.100155

print(
    "Registros:",
    len(glaucoma_labels)
)
print(
    "Negativos:",
    int((glaucoma_labels == 0).sum())
)
print(
    "Positivos:",
    int((glaucoma_labels == 1).sum())
)
print(
    "Score mínimo:",
    glaucoma_scores.min()
)
print(
    "Score máximo:",
    glaucoma_scores.max()
)
print(
    "Umbral bruto:",
    GLAUCOMA_RAW_THRESHOLD
)

Registros: 202
Negativos: 175
Positivos: 27
Score mínimo: 0.00020094075
Score máximo: 0.9674187
Umbral bruto: 0.100155


### 7.2. Conversión de scores sigmoid a logits (Log-Odds)

Se acotan numéricamente los scores continuos para calcular los logits equivalentes mediante la función logit inversa.

In [12]:
EPSILON = 1e-7
glaucoma_scores_clipped = np.clip(
    glaucoma_scores,
    EPSILON,
    1.0 - EPSILON
)
glaucoma_logits = np.log(
    glaucoma_scores_clipped
    / (1.0 - glaucoma_scores_clipped)
)

print(
    "Logit mínimo:",
    glaucoma_logits.min()
)
print(
    "Logit máximo:",
    glaucoma_logits.max()
)

Logit mínimo: -8.512299508538325
Logit máximo: 3.3908928862073036


### 7.3. Parametrización monótona de Platt Scaling y optimización de parámetros

Se define una parametrización con pendiente positiva obligatoria para garantizar isotonicidad estricta y se optimizan los parámetros minimizando la pérdida logarítmica binaria.

In [13]:
def platt_probabilities(
    logits,
    log_a,
    b
):
    a = np.exp(log_a)
    return expit(
        a * logits + b
    )


def platt_objective(
    parameters
):
    log_a, b = parameters
    calibrated = platt_probabilities(
        glaucoma_logits,
        log_a,
        b
    )
    calibrated = np.clip(
        calibrated,
        EPSILON,
        1.0 - EPSILON
    )
    return log_loss(
        glaucoma_labels,
        calibrated
    )


platt_result = minimize(
    platt_objective,
    x0=np.array([
        0.0,
        0.0
    ]),
    method="BFGS"
)

GLAUCOMA_LOG_A = float(
    platt_result.x[0]
)
GLAUCOMA_A = float(
    np.exp(
        GLAUCOMA_LOG_A
    )
)
GLAUCOMA_B = float(
    platt_result.x[1]
)

print(
    "Optimización exitosa:",
    platt_result.success
)
print(
    "A:",
    GLAUCOMA_A
)
print(
    "B:",
    GLAUCOMA_B
)

Optimización exitosa: True
A: 0.7882374024841463
B: -0.5499764599494648


### 7.4. Estimación de probabilidades calibradas y verificación de mejora en NLL

Se generan las probabilidades calibradas para Validation y se evalúa la reducción de la pérdida logarítmica binaria respecto al score original.

In [14]:
glaucoma_calibrated_probability = (
    platt_probabilities(
        glaucoma_logits,
        GLAUCOMA_LOG_A,
        GLAUCOMA_B
    )
)

glaucoma_nll_before = log_loss(
    glaucoma_labels,
    np.column_stack(
        [
            1.0 - glaucoma_scores,
            glaucoma_scores
        ]
    )
)

glaucoma_nll_after = log_loss(
    glaucoma_labels,
    np.column_stack(
        [
            1.0 - glaucoma_calibrated_probability,
            glaucoma_calibrated_probability
        ]
    )
)

print(
    f"NLL antes:   {glaucoma_nll_before:.6f}"
)

print(
    f"NLL después: {glaucoma_nll_after:.6f}"
)

print(
    "Probabilidades calibradas:",
    glaucoma_calibrated_probability.shape
)

NLL antes:   0.274708
NLL después: 0.265096
Probabilidades calibradas: (202,)


### 7.5. Transformación del umbral operativo y verificación de equivalencia de decisiones

Se traslada el umbral operativo bruto (0.100155) al espacio de probabilidad calibrada y se demuestra que las decisiones clasificación binarias se conservan de forma idéntica en el 100% de los casos.

In [15]:
threshold_clipped = np.clip(
    GLAUCOMA_RAW_THRESHOLD,
    EPSILON,
    1.0 - EPSILON
)
threshold_logit = np.log(
    threshold_clipped
    / (1.0 - threshold_clipped)
)
GLAUCOMA_CALIBRATED_THRESHOLD = float(
    platt_probabilities(
        np.array([
            threshold_logit
        ]),
        GLAUCOMA_LOG_A,
        GLAUCOMA_B
    )[0]
)

print(
    "Umbral bruto:",
    GLAUCOMA_RAW_THRESHOLD
)
print(
    "Umbral calibrado:",
    GLAUCOMA_CALIBRATED_THRESHOLD
)

Umbral bruto: 0.100155
Umbral calibrado: 0.0927462872218843


### 7.5.1. Comprobación de preservación de decisiones clasificación

Se comprueba la equivalencia lógica estricta entre la clasificación con el score original y la clasificación bajo la probabilidad calibrada.

In [16]:
raw_predictions = (
    glaucoma_scores
    >= GLAUCOMA_RAW_THRESHOLD
).astype(int)
calibrated_predictions = (
    glaucoma_calibrated_probability
    >= GLAUCOMA_CALIBRATED_THRESHOLD
).astype(int)

print(
    "Decisiones preservadas:",
    np.array_equal(
        raw_predictions,
        calibrated_predictions
    )
)
print(
    "Alto riesgo antes:",
    int(raw_predictions.sum())
)
print(
    "Alto riesgo después:",
    int(calibrated_predictions.sum())
)

Decisiones preservadas: True
Alto riesgo antes: 77
Alto riesgo después: 77


### 7.6. Probabilidad calibrada asociada a la categoría decidida

Se calcula la probabilidad calibrada asociada a la categoría decidida. Este valor se conserva como información probabilística del modelo. No obstante, debido a que la clasificación de Glaucoma utiliza un umbral operativo asimétrico, este valor no se utiliza como índice definitivo de confianza para la política `Requiere revisión`. La confianza de decisión utilizada por dicha política se define posteriormente como la distancia normalizada respecto al umbral operativo.

In [17]:
glaucoma_class_probability = np.where(
    calibrated_predictions == 1,
    glaucoma_calibrated_probability,
    1.0 - glaucoma_calibrated_probability
)

print(
    "Confianza mínima:",
    glaucoma_class_probability.min()
)
print(
    "Confianza máxima:",
    glaucoma_class_probability.max()
)

Confianza mínima: 0.09437940749557847
Confianza máxima: 0.9992971678049595


## 8. Persistencia del calibrador de Glaucoma

Se guardan los coeficientes logísticos \(a\) y \(b\), el umbral operativo calibrado y las estadísticas de validación en `results/calibration/glaucoma_platt_scaling.json`.

In [18]:
GLAUCOMA_CALIBRATION_PATH = (
    CALIBRATION_RESULTS_DIR
    / "glaucoma_platt_scaling.json"
)

glaucoma_calibration_config = {
    "task":
        "glaucoma",
    "model":
        "ResNet50V2 Fine-Tuning",
    "method":
        "platt_scaling",
    "a":
        float(GLAUCOMA_A),
    "b":
        float(GLAUCOMA_B),
    "log_a":
        float(GLAUCOMA_LOG_A),
    "raw_threshold":
        float(GLAUCOMA_RAW_THRESHOLD),
    "calibrated_threshold":
        float(GLAUCOMA_CALIBRATED_THRESHOLD),
    "validation_samples":
        int(len(glaucoma_labels)),
    "negative_samples":
        int(
            (glaucoma_labels == 0).sum()
        ),
    "positive_samples":
        int(
            (glaucoma_labels == 1).sum()
        ),
    "nll_before":
        float(glaucoma_nll_before),
    "nll_after":
        float(glaucoma_nll_after),
    "labels": {
        "0": "Bajo riesgo de Glaucoma",
        "1": "Alto riesgo de Glaucoma"
    }
}

with open(
    GLAUCOMA_CALIBRATION_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        glaucoma_calibration_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Calibrador Glaucoma guardado en:",
    GLAUCOMA_CALIBRATION_PATH
)

Calibrador Glaucoma guardado en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/calibration/glaucoma_platt_scaling.json


### 8.1. Verificación final de persistencia de calibradores

Se comprueba la existencia efectiva en disco de los dos archivos de configuración de calibración antes de cerrar el notebook.

In [19]:
print(
    "RD calibrador existe:",
    RD_CALIBRATION_PATH.exists()
)
print(
    "Glaucoma calibrador existe:",
    GLAUCOMA_CALIBRATION_PATH.exists()
)

RD calibrador existe: True
Glaucoma calibrador existe: True


## 9. Definición del criterio de baja confianza

La advertencia `Requiere revisión` no se asigna mediante un porcentaje
arbitrario.

El umbral se determina exclusivamente sobre Validation utilizando los
errores conocidos de cada modelo.

Una predicción incorrecta se considera un caso que idealmente debería
ser identificado como de baja confianza.

Para cada posible umbral se calcula:

- porcentaje de casos enviados a revisión;
- precisión para identificar errores;
- recall de errores detectados;
- F1 para detección de errores;
- accuracy de las predicciones que no requieren revisión.

El umbral seleccionado maximiza el F1 para la identificación de errores.
En caso de empate se favorece el umbral que envía menos casos a revisión.

### 9.1. Análisis de confianza y cuantificación de errores en Retinopatía Diabética

Se determinan las predicciones categóricas, el nivel de confianza máxima obtenido de los scores calibrados y los errores efectivos frente a las etiquetas de referencia sobre Validation.

In [20]:
rd_predictions = np.argmax(
    rd_calibrated_scores,
    axis=1
)
rd_confidence = np.max(
    rd_calibrated_scores,
    axis=1
)
rd_errors = (
    rd_predictions
    != rd_labels
).astype(int)

print(
    "Predicciones RD:",
    len(rd_predictions)
)
print(
    "Correctas:",
    int((rd_errors == 0).sum())
)
print(
    "Incorrectas:",
    int((rd_errors == 1).sum())
)
print(
    "Accuracy:",
    float(
        (rd_errors == 0).mean()
    )
)
print(
    "Confianza mínima:",
    rd_confidence.min()
)
print(
    "Confianza máxima:",
    rd_confidence.max()
)

Predicciones RD: 523
Correctas: 408
Incorrectas: 115
Accuracy: 0.780114722753346
Confianza mínima: 0.29256778381211507
Confianza máxima: 0.9997385940668643


### 9.2. Función para evaluar umbrales de revisión clínica

Implementación de una función genérica de barrido para evaluar métricas diagnósticas y operativas asociadas a la política de derivación a revisión según el nivel de confianza predicho.

In [21]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

def evaluate_review_thresholds(
    confidence,
    errors,
    predictions,
    labels,
    thresholds=None
):
    if thresholds is None:
        thresholds = np.linspace(
            0.0,
            1.0,
            101
        )
    rows = []
    for threshold in thresholds:
        review = (
            confidence < threshold
        ).astype(int)
        review_rate = float(
            review.mean()
        )
        error_precision = precision_score(
            errors,
            review,
            zero_division=0
        )
        error_recall = recall_score(
            errors,
            review,
            zero_division=0
        )
        error_f1 = f1_score(
            errors,
            review,
            zero_division=0
        )
        accepted = (
            review == 0
        )
        if accepted.sum() > 0:
            accepted_accuracy = float(
                (
                    predictions[accepted]
                    == labels[accepted]
                ).mean()
            )
        else:
            accepted_accuracy = np.nan
        rows.append(
            {
                "threshold":
                    threshold,
                "review_rate":
                    review_rate,
                "error_precision":
                    error_precision,
                "error_recall":
                    error_recall,
                "error_f1":
                    error_f1,
                "accepted_accuracy":
                    accepted_accuracy,
                "reviewed_cases":
                    int(review.sum()),
                "accepted_cases":
                    int(accepted.sum())
            }
        )
    return pd.DataFrame(rows)

## 10. Búsqueda y selección del umbral de revisión para Retinopatía Diabética

Se calcula el rendimiento de la política de remisión para distintos umbrales de confianza en RD, identificando los 10 mejores candidatos bajo el criterio jerárquico de máximo F1 de detección de errores y menor tasa de casos derivados.

In [22]:
rd_threshold_analysis = (
    evaluate_review_thresholds(
        confidence=rd_confidence,
        errors=rd_errors,
        predictions=rd_predictions,
        labels=rd_labels
    )
)

rd_best_candidates = (
    rd_threshold_analysis
    .sort_values(
        by=[
            "error_f1",
            "review_rate"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(10)
)
display(
    rd_best_candidates.round(4)
)

,threshold,review_rate,error_precision,error_recall,error_f1,accepted_accuracy,reviewed_cases,accepted_cases
83,0.83,0.5067,0.3962,0.9130,0.5526,0.9612,265,258
84,0.84,0.5143,0.3941,0.9217,0.5521,0.9646,269,254
77,0.77,0.4532,0.4093,0.8435,0.5511,0.9371,237,286
82,0.82,0.5029,0.3954,0.9043,0.5503,0.9577,263,260
75,0.75,0.4417,0.4113,0.8261,0.5491,0.9315,231,292
93,0.93,0.5813,0.3783,1.0000,0.5489,1.0000,304,219
76,0.76,0.4493,0.4085,0.8348,0.5486,0.9340,235,288
92,0.92,0.5679,0.3805,0.9826,0.5485,0.9912,297,226
88,0.88,0.5411,0.3852,0.9478,0.5477,0.9750,283,240
73,0.73,0.4302,0.4133,0.8087,0.5471,0.9262,225,298


### 10.1. Selección preliminar del umbral óptimo de revisión para RD

Se extraen las estadísticas operativas del umbral candidato que maximiza el balance entre detección de errores y conservación de cobertura automática de resultados.

In [23]:
rd_best_row = (
    rd_best_candidates.iloc[0]
)
RD_REVIEW_THRESHOLD = float(
    rd_best_row["threshold"]
)
print(
    "Umbral RD candidato:",
    RD_REVIEW_THRESHOLD
)
print(
    "Casos enviados a revisión:",
    int(
        rd_best_row[
            "reviewed_cases"
        ]
    )
)
print(
    "Porcentaje enviado a revisión:",
    f"{rd_best_row['review_rate'] * 100:.2f}%"
)
print(
    "Errores detectados (recall):",
    f"{rd_best_row['error_recall'] * 100:.2f}%"
)
print(
    "Accuracy de casos no enviados a revisión:",
    f"{rd_best_row['accepted_accuracy'] * 100:.2f}%"
)

Umbral RD candidato: 0.8300000000000001
Casos enviados a revisión: 265
Porcentaje enviado a revisión: 50.67%
Errores detectados (recall): 91.30%
Accuracy de casos no enviados a revisión: 96.12%


## 11. Confianza de decisión para Glaucoma

A diferencia de la clasificación multiclase, la inferencia binaria de Glaucoma se rige por un umbral asimétrico congelado (\(t = 0.092746\)).

La estabilidad de la decisión clínica depende de la distancia relativa entre la probabilidad calibrada \(p\) y dicho umbral:

- \(p \approx t\): Decisión en zona de máxima incertidumbre de la decisión.
- \(p \gg t\) o \(p \ll t\): Decisión nítida y alejada de la frontera operativa.

Se define un índice de confianza de decisión normalizado en \([0.0, 1.0]\) que cuantifica dicha distancia relativa a ambos lados del umbral.

In [24]:
glaucoma_decision_confidence = np.where(
    glaucoma_calibrated_probability
    >= GLAUCOMA_CALIBRATED_THRESHOLD,
    (
        glaucoma_calibrated_probability
        - GLAUCOMA_CALIBRATED_THRESHOLD
    )
    /
    (
        1.0
        - GLAUCOMA_CALIBRATED_THRESHOLD
    ),
    (
        GLAUCOMA_CALIBRATED_THRESHOLD
        - glaucoma_calibrated_probability
    )
    /
    GLAUCOMA_CALIBRATED_THRESHOLD
)

glaucoma_decision_confidence = np.clip(
    glaucoma_decision_confidence,
    0.0,
    1.0
)
glaucoma_predictions = (
    glaucoma_calibrated_probability
    >= GLAUCOMA_CALIBRATED_THRESHOLD
).astype(int)
glaucoma_errors = (
    glaucoma_predictions
    != glaucoma_labels
).astype(int)

print(
    "Predicciones Glaucoma:",
    len(glaucoma_predictions)
)
print(
    "Correctas:",
    int(
        (glaucoma_errors == 0).sum()
    )
)
print(
    "Incorrectas:",
    int(
        (glaucoma_errors == 1).sum()
    )
)
print(
    "Accuracy:",
    float(
        (glaucoma_errors == 0).mean()
    )
)
print(
    "Confianza mínima:",
    glaucoma_decision_confidence.min()
)
print(
    "Confianza máxima:",
    glaucoma_decision_confidence.max()
)

Predicciones Glaucoma: 202
Correctas: 148
Incorrectas: 54
Accuracy: 0.7326732673267327
Confianza mínima: 1.3489455993701003e-06
Confianza máxima: 0.9924219910457533


## 12. Búsqueda y selección del umbral de revisión para Glaucoma

Se evalúa el barrido de umbrales sobre el índice de confianza de decisión de Glaucoma, seleccionando los candidatos que maximizan la identificación de clasificaciones dudosas o erróneas.

In [25]:
glaucoma_threshold_analysis = (
    evaluate_review_thresholds(
        confidence=
            glaucoma_decision_confidence,
        errors=
            glaucoma_errors,
        predictions=
            glaucoma_predictions,
        labels=
            glaucoma_labels
    )
)

glaucoma_best_candidates = (
    glaucoma_threshold_analysis
    .sort_values(
        by=[
            "error_f1",
            "review_rate"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(10)
)
display(
    glaucoma_best_candidates.round(4)
)

,threshold,review_rate,error_precision,error_recall,error_f1,accepted_accuracy,reviewed_cases,accepted_cases
29,0.29,0.3465,0.6571,0.8519,0.7419,0.9394,70,132
30,0.30,0.3465,0.6571,0.8519,0.7419,0.9394,70,132
31,0.31,0.3515,0.6479,0.8519,0.7360,0.9389,71,131
24,0.24,0.2871,0.7069,0.7593,0.7321,0.9097,58,144
28,0.28,0.3416,0.6522,0.8333,0.7317,0.9323,69,133
25,0.25,0.3020,0.6885,0.7778,0.7304,0.9149,61,141
32,0.32,0.3564,0.6389,0.8519,0.7302,0.9385,72,130
22,0.22,0.2772,0.7143,0.7407,0.7273,0.9041,56,146
23,0.23,0.2772,0.7143,0.7407,0.7273,0.9041,56,146
27,0.27,0.3317,0.6567,0.8148,0.7273,0.9259,67,135


### 12.1. Selección preliminar del umbral óptimo de revisión para Glaucoma

Extracción y análisis de las métricas asociadas al umbral candidato para derivación clínica por baja confianza en Glaucoma.

In [26]:
glaucoma_best_row = (
    glaucoma_best_candidates.iloc[0]
)
GLAUCOMA_REVIEW_THRESHOLD = float(
    glaucoma_best_row["threshold"]
)
print(
    "Umbral Glaucoma candidato:",
    GLAUCOMA_REVIEW_THRESHOLD
)
print(
    "Casos enviados a revisión:",
    int(
        glaucoma_best_row[
            "reviewed_cases"
        ]
    )
)
print(
    "Porcentaje enviado a revisión:",
    f"{glaucoma_best_row['review_rate'] * 100:.2f}%"
)
print(
    "Errores detectados (recall):",
    f"{glaucoma_best_row['error_recall'] * 100:.2f}%"
)
print(
    "Accuracy de casos no enviados a revisión:",
    f"{glaucoma_best_row['accepted_accuracy'] * 100:.2f}%"
)

Umbral Glaucoma candidato: 0.29
Casos enviados a revisión: 70
Porcentaje enviado a revisión: 34.65%
Errores detectados (recall): 85.19%
Accuracy de casos no enviados a revisión: 93.94%


## 13. Política definitiva de revisión por baja confianza

Los umbrales de baja confianza fueron determinados exclusivamente sobre
Validation.

Una predicción incorrecta fue considerada un caso que idealmente debería
ser identificado para revisión. Se evaluaron múltiples umbrales y se
seleccionó aquel que maximizó el F1 para detección de errores, favoreciendo
una menor tasa de revisión en caso de resultados equivalentes.

Los umbrales congelados son:

- Retinopatía Diabética: 0.83.
- Glaucoma: 0.29.

Para Retinopatía Diabética, la confianza corresponde a la probabilidad
calibrada asignada a la clase predicha.

Para Glaucoma, la confianza corresponde a un índice normalizado de
distancia respecto al umbral operativo de decisión. Este índice representa
estabilidad de la decisión y no debe interpretarse como probabilidad
clínica de glaucoma.

Estos umbrales únicamente controlan la advertencia `Requiere revisión`.
No modifican la clase o categoría predicha por los modelos.

In [27]:
REVIEW_POLICY_PATH = (
    CALIBRATION_RESULTS_DIR
    / "review_policy.json"
)

review_policy = {
    "diabetic_retinopathy": {
        "review_threshold":
            float(RD_REVIEW_THRESHOLD),

        "confidence_type":
            "calibrated_predicted_class_probability",

        "validation_samples":
            int(len(rd_labels)),

        "validation_errors":
            int(rd_errors.sum()),

        "reviewed_cases":
            int(rd_best_row["reviewed_cases"]),

        "review_rate":
            float(rd_best_row["review_rate"]),

        "error_recall":
            float(rd_best_row["error_recall"]),

        "accepted_accuracy":
            float(rd_best_row["accepted_accuracy"])
    },

    "glaucoma": {
        "review_threshold":
            float(GLAUCOMA_REVIEW_THRESHOLD),

        "confidence_type":
            "normalized_decision_distance",

        "validation_samples":
            int(len(glaucoma_labels)),

        "validation_errors":
            int(glaucoma_errors.sum()),

        "reviewed_cases":
            int(
                glaucoma_best_row[
                    "reviewed_cases"
                ]
            ),

        "review_rate":
            float(
                glaucoma_best_row[
                    "review_rate"
                ]
            ),

        "error_recall":
            float(
                glaucoma_best_row[
                    "error_recall"
                ]
            ),

        "accepted_accuracy":
            float(
                glaucoma_best_row[
                    "accepted_accuracy"
                ]
            )
    }
}

with open(
    REVIEW_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        review_policy,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Política de revisión guardada en:",
    REVIEW_POLICY_PATH
)

Política de revisión guardada en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/calibration/review_policy.json
